In [74]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('espn_cfb_2026.db')

pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,Teams
1,Games
2,Box_Scores
3,Player_Box_Scores
4,Raw_Boxscores
5,Scoreboard_Fetches
6,Player_Boxscore_Fetches
7,Team_Defensive_Stats
8,Team_Defense_Opponent_Stats
9,Raw_Plays


In [75]:
load_games = """
SELECT game_id, week, home_team_id, away_team_id, home_score, away_score, neutral_site
FROM Games
WHERE status = 'Final'
"""

games = pd.read_sql_query(load_games, conn)

print(games.shape)
display(games.head())
display(games.dtypes)

(752, 7)


,game_id,week,home_team_id,away_team_id,home_score,away_score,neutral_site
0,401864494,1,30,23,42,26,0
1,401856766,1,2628,153,10,15,1
2,401858202,1,258,152,34,8,0
3,401864577,1,2449,55,33,7,0
4,401866408,1,2199,16,28,17,0


game_id           str
week            int64
home_team_id      str
away_team_id      str
home_score      int64
away_score      int64
neutral_site    int64
dtype: object

In [76]:
load_teams = """
SELECT team_id, short_name, conference
FROM Teams
"""

teams = pd.read_sql_query(load_teams, conn)

teams["conference"].value_counts(dropna=False)

conference
NaN                          102
Big Ten Conference            18
Atlantic Coast Conference     17
Big 12 Conference             16
Southeastern Conference       16
American Conference           14
Sun Belt Conference           14
Mid-American Conference       13
Conference USA                10
Mountain West Conference      10
Pac-12 Conference              8
FBS Independents               2
Name: count, dtype: int64

In [77]:
fcs_ids = set(teams[teams["conference"].isna()]["team_id"])

games["home_team_id"] = games["home_team_id"].mask(games["home_team_id"].isin(fcs_ids), "FCS")
games["away_team_id"] = games["away_team_id"].mask(games["away_team_id"].isin(fcs_ids), "FCS")

new_row = pd.DataFrame([{"team_id": "FCS", "short_name": "FCS", "conference": "FCS"}])
teams = teams if "FCS" in teams["team_id"].values else pd.concat([teams, new_row], ignore_index=True)

teams.shape

(241, 3)

In [78]:
home_fcs = games["home_team_id"].eq("FCS")
away_fcs = games["away_team_id"].eq("FCS")

print("Home FCS games:", home_fcs.sum())
print("Away FCS games:", away_fcs.sum())
print("Games vs. FCS (either side):", (home_fcs | away_fcs).sum())
print("FCS vs. FCS (both sides):", (home_fcs & away_fcs).sum())

Home FCS games: 0
Away FCS games: 123
Games vs. FCS (either side): 123
FCS vs. FCS (both sides): 0


## TODO: Cell 6: Build `team_games` (one row per team per game)

### The answer to the rename question
Each copy needs its columns renamed into a **shared, team-relative vocabulary**. Then the two copies line up column for column when you stack them:

| Original column | Home team's view | Away team's view |
|---|---|---|
| `home_team_id` | → `team_id` | → `opp_id` |
| `away_team_id` | → `opp_id` | → `team_id` |
| `home_score` | → `points_for` | → `points_against` |
| `away_score` | → `points_against` | → `points_for` |

The two views use **the same renames, flipped**. `game_id` and `week` don't change.

### Step A: home view
- Make a copy of `games` and call `.rename(columns={...})` with the left-hand mapping above. `rename` takes a dictionary of `{"old_name": "new_name"}`.
- Add a `location` column. It's `"home"`, **unless** `neutral_site == 1`, in which case it's `"neutral"`. Use `np.where(condition, value_if_true, value_if_false)`, which is the vectorized version of if/else.

### Step B: away view
- Same idea, using the right-hand mapping.
- Its `location` is `"away"` unless the game is at a neutral site.

### Step C: stack them
- `pd.concat([home_view, away_view], ignore_index=True)` → `team_games`.
- Drop `neutral_site`. `location` now holds that information.

### Step D: add two derived columns
- `margin` = `points_for − points_against`
- `win` = 1 if `margin > 0`, else 0. (College football has no ties since overtime was added in 1996. But check it with the math below rather than assuming.)

### Why location matters
Later, the Massey rating will estimate **home-field advantage** (usually about 2–3 points in college football) and remove it from each team's margins. Beating someone by 3 at home is less impressive than winning by 3 on the road. Neutral-site games get no adjustment.

### Checks (new cell)
1. `len(team_games)` should be **1504**, exactly 2 × 752.
2. Each `game_id` should appear **exactly twice**. Use `team_games["game_id"].value_counts()` and check its `.unique()` values.
3. `team_games["margin"].sum()` should be **0**. Each game's +margin and −margin cancel out. If it isn't 0, a rename got crossed somewhere. This is a good sanity check for the whole reshape.
4. `(team_games["margin"] == 0).sum()` should be **0**, which confirms there are no ties.

Write Steps A–D. If you get stuck partway, send me what you have.